# 🔥 Zira Training Launcher

Production-grade decoder-only Transformer optimised for **Google Colab Free TPU (v5e-1)**.

| Model | Params | Use Case |
|-------|--------|---------|
| Zira-Micro   | ~30M  | Fast debug runs (<2 hrs) |
| Zira-Small   | ~60M  | Full SFT in one Colab session |
| Zira-Compact | ~80M  | Balanced pretrain + SFT |
| Zira-Base    | ~200M | Foundation pretrain across sessions |
| Zira-Pro     | ~450M | Near full TPU utilisation |


## 1. Environment setup

In [ ]:
import os
import subprocess
import sys

def run(cmd):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    return result.returncode

# ── Detect runtime ──────────────────────────────────────────────────────────
print('Python:', sys.version)

try:
    import torch_xla  # type: ignore
    print('torch_xla already available:', torch_xla.__version__)
    HAS_XLA = True
except ImportError:
    print('torch_xla not found – attempting install …')
    # Install the Colab-compatible nightly wheel
    run('pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html -q')
    try:
        import torch_xla  # type: ignore
        HAS_XLA = True
        print('torch_xla installed:', torch_xla.__version__)
    except ImportError:
        HAS_XLA = False
        print('WARNING: torch_xla not available – falling back to GPU/CPU')

import torch
print('PyTorch:', torch.__version__)

if HAS_XLA:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print('TPU device:', device)
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print('GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device('cpu')
    print('Running on CPU (debug mode)')

## 2. Install dependencies

In [ ]:
run('pip install datasets transformers sentencepiece -q')

## 3. Clone the Zira repository

In [ ]:
REPO_URL = 'https://github.com/dill-lk/Zira.git'  # update if forked
WORK_DIR = '/content/Zira'

if not os.path.isdir(WORK_DIR):
    run(f'git clone {REPO_URL} {WORK_DIR}')
else:
    run(f'cd {WORK_DIR} && git pull')

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())

## 4. (Optional) Mount Google Drive for persistent checkpoints

In [ ]:
USE_DRIVE = True  # set False to skip

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/zira/checkpoints'
    os.makedirs(CKPT_DIR, exist_ok=True)
    print('Checkpoint dir:', CKPT_DIR)
else:
    CKPT_DIR = f'{WORK_DIR}/checkpoints'
    os.makedirs(CKPT_DIR, exist_ok=True)

## 5. Configure training

In [ ]:
# ── Choose your model ────────────────────────────────────────────────────────
# Options: 'micro' | 'small' | 'compact' | 'base' | 'pro'
MODEL_SIZE   = 'micro'

# ── Dataset ───────────────────────────────────────────────────────────────────
# Pre-train: 'wikitext-103' | 'openwebtext'
PRETRAIN_DS  = 'wikitext-103'

# SFT:       'tatsu-lab/alpaca' | 'HuggingFaceH4/ultrachat_200k'
SFT_DS       = 'tatsu-lab/alpaca'

# ── Tokenizer ─────────────────────────────────────────────────────────────────
# Any HuggingFace tokenizer; llama tokenizer is a sensible default
TOKENIZER    = 'huggyllama/llama-7b'

# ── Training hyperparams ──────────────────────────────────────────────────────
BATCH_SIZE   = 8        # per-device batch size
MAX_STEPS    = 10000    # set None to use config default
SFT_STEPS    = 2000

# ── Checkpoint ────────────────────────────────────────────────────────────────
RESUME_CKPT  = ''       # path to resume from, or '' to start fresh
PRETRAIN_CKPT = ''      # path to pretrain ckpt to load before SFT

print('Configuration set.')

## 6a. Phase 1 – Foundation Pre-training

In [ ]:
from zira.config import get_config
from zira.train_pretrain import train
import argparse

cfg = get_config(MODEL_SIZE)
cfg.checkpoint_dir = CKPT_DIR
if MAX_STEPS:
    cfg.max_steps = MAX_STEPS

args = argparse.Namespace(
    model        = MODEL_SIZE,
    dataset      = PRETRAIN_DS,
    tokenizer    = TOKENIZER,
    batch_size   = BATCH_SIZE,
    steps        = MAX_STEPS,
    log_every    = 50,
    resume       = RESUME_CKPT,
    use_drive    = USE_DRIVE,
    max_samples  = None,
    num_workers  = 2,
    tpu_spawn    = False,
)

train(cfg, args)

## 6b. Phase 2 – Supervised Fine-Tuning (SFT)

In [ ]:
from zira.train_sft import train_sft
import argparse

sft_cfg = get_config(MODEL_SIZE)
sft_cfg.checkpoint_dir = CKPT_DIR

sft_args = argparse.Namespace(
    model          = MODEL_SIZE,
    dataset        = SFT_DS,
    tokenizer      = TOKENIZER,
    pretrain_ckpt  = PRETRAIN_CKPT,
    batch_size     = BATCH_SIZE,
    sft_steps      = SFT_STEPS,
    log_every      = 50,
    resume         = '',
    use_drive      = USE_DRIVE,
    max_samples    = None,
    num_workers    = 2,
    tpu_spawn      = False,
)

train_sft(sft_cfg, sft_args)

## 7. Generate text

In [ ]:
from zira.model import ZiraModel
from zira.tokenizer import ZiraTokenizer
from zira.generate import generate
from zira.utils import find_latest_checkpoint, load_checkpoint

ckpt_path = find_latest_checkpoint(CKPT_DIR)
if ckpt_path is None:
    print('No checkpoint found – run training first.')
else:
    gen_cfg = get_config(MODEL_SIZE)
    gen_model = ZiraModel(gen_cfg).to(device)
    load_checkpoint(ckpt_path, gen_model, map_location=str(device))
    gen_tok = ZiraTokenizer(hf_name=TOKENIZER)

    prompt = 'Once upon a time in a land far away'
    output = generate(
        model          = gen_model,
        tokenizer      = gen_tok,
        prompt         = prompt,
        max_new_tokens = 200,
        temperature    = 0.8,
        top_k          = 50,
        top_p          = 0.95,
        device         = device,
    )
    print(output)